1. Instalação das dependências

In [2]:
!apt-get update -qq
!apt-get install -y swi-prolog

!pip install pyswip pandas scikit-learn

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
swi-prolog is already the newest version (8.4.2+dfsg-2ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 10 not upgraded.


2. Importação das bibliotecas

In [3]:
import pandas as pd

from pyswip import Prolog

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

3. Carregamento da base Prolog

In [4]:
!wget -O rede_social.pl "https://raw.githubusercontent.com/victorlfrazao/AprendizadoRelacionalEstatistico/refs/heads/main/rede_social.pl"

--2026-05-20 01:56:51--  https://raw.githubusercontent.com/victorlfrazao/AprendizadoRelacionalEstatistico/refs/heads/main/rede_social.pl
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1143 (1.1K) [text/plain]
Saving to: ‘rede_social.pl’

rede_social.pl      100%[===================>]   1.12K  --.-KB/s    in 0s      

2026-05-20 01:56:51 (66.6 MB/s) - ‘rede_social.pl’ saved [1143/1143]



In [6]:
prolog = Prolog()

prolog.consult("rede_social.pl")

print("Base Prolog carregada!")

Base Prolog carregada!


4. Leitura do dataset financeiro

In [5]:
!wget -O dados_financeiros.csv "https://raw.githubusercontent.com/victorlfrazao/AprendizadoRelacionalEstatistico/refs/heads/main/dados_financeiros.csv"

--2026-05-20 01:56:51--  https://raw.githubusercontent.com/victorlfrazao/AprendizadoRelacionalEstatistico/refs/heads/main/dados_financeiros.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 298 [text/plain]
Saving to: ‘dados_financeiros.csv’

dados_financeiros.c 100%[===================>]     298  --.-KB/s    in 0s      

2026-05-20 01:56:51 (5.55 MB/s) - ‘dados_financeiros.csv’ saved [298/298]



In [7]:
df = pd.read_csv(
    "dados_financeiros.csv"
)

df

,cliente_id,renda_mensal,score_classico,inadimplente_historico
0,alice,7200,780,0
1,pedro,4300,640,0
2,renata,3500,590,0
3,gustavo,1600,390,1
4,camila,5100,710,0
5,eduardo,2900,540,0
6,sergio,2600,500,0
7,helena,1800,420,1
8,victor,8400,820,0
9,larissa,3900,610,0


5. Identificação dos inadimplentes

In [8]:
inadimplentes = list(
    prolog.query(
        "inadimplente(X)"
    )
)

inadimplentes = [
    item["X"]
    for item in inadimplentes
]

print(inadimplentes)

['gustavo', 'helena']


6. Função de cálculo do grau de risco

In [9]:
def obter_grau_risco(nome):

    menor_grau = 999

    for alvo in inadimplentes:

        if nome == alvo:
            return 0

        consulta = list(
            prolog.query(
                f"risco_conexao({nome}, {alvo}, G)"
            )
        )

        if consulta:

            grau = min(
                r["G"]
                for r in consulta
            )

            menor_grau = min(
                menor_grau,
                grau
            )

    return menor_grau

7. Criação da feature lógica

In [10]:
df["grau_risco_rede"] = (
    df["cliente_id"]
    .apply(obter_grau_risco)
)

df

,cliente_id,renda_mensal,score_classico,inadimplente_historico,grau_risco_rede
0,alice,7200,780,0,3
1,pedro,4300,640,0,2
2,renata,3500,590,0,1
3,gustavo,1600,390,1,0
4,camila,5100,710,0,3
5,eduardo,2900,540,0,2
6,sergio,2600,500,0,1
7,helena,1800,420,1,0
8,victor,8400,820,0,3
9,larissa,3900,610,0,1


8. Preparação das variáveis do modelo

In [11]:
X = df[
    [
        "renda_mensal",
        "score_classico",
        "grau_risco_rede"
    ]
]

y = df[
    "inadimplente_historico"
]

9. Normalização dos dados

In [12]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

10. Treinamento da regressão logística

In [13]:
modelo = LogisticRegression()

modelo.fit(
    X_scaled,
    y
)

print("Modelo treinado!")

Modelo treinado!


11. Análise dos coeficientes

In [14]:
coeficientes = pd.DataFrame({
    "feature": X.columns,
    "peso": modelo.coef_[0]
})

coeficientes

,feature,peso
0,renda_mensal,-0.471625
1,score_classico,-0.813844
2,grau_risco_rede,-0.965769


12. Predição de um novo cliente

In [15]:
novo_cliente = [[3000, 560, 2]]

novo_cliente_scaled = (
    scaler.transform(
        novo_cliente
    )
)

prob = modelo.predict_proba(
    novo_cliente_scaled
)[0][1]

print(
    f"{prob:.2f} :: risco(cliente_novo)"
)

0.05 :: risco(cliente_novo)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


13. Regra probabilística explicável

In [16]:
nome_cliente = "cliente_novo"

grau = novo_cliente[0][2]

print(
    f"{prob:.2f}::risco({nome_cliente}) :-"
)

print(
    f"    risco_conexao("
    f"{nome_cliente}, "
    f"gustavo, "
    f"{grau}"
    f")."
)

0.05::risco(cliente_novo) :-
    risco_conexao(cliente_novo, gustavo, 2).
